# Qwen Conversation Path Leakage Tests

This notebook tests whether abandoned or overwritten conversation paths leak into the final answer.

Controls:
- `temperature = 0`
- same system prompt for every run
- fresh chat for every Chat A / Chat B condition
- generated intermediate assistant replies are kept in Chat B history
- local Qwen runtime, no memory/router/adapter code involved

System prompt:

> You are a precise assistant. Follow the latest user instruction exactly. Ignore abandoned or explicitly canceled earlier requests.


In [ ]:
# Cell 1: Repo bootstrap, imports, and config
from __future__ import annotations

import gc
import importlib.metadata as importlib_metadata
import inspect
import json
import os
import re
import subprocess
import sys
import time
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any


REPO_URL = "https://github.com/tungooxx/CogMem.git"


def find_or_create_repo_root() -> Path:
    candidates = [Path.cwd(), Path("/notebooks/CogMem"), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "cogmem").is_dir():
            return candidate.resolve()
    if Path("/notebooks").exists():
        target = Path("/notebooks/CogMem")
        if not target.exists():
            subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
        if (target / "cogmem").is_dir():
            return target.resolve()
    raise RuntimeError("Could not find CogMem repo. Run this notebook from the repo root or clone it to /notebooks/CogMem.")


REPO_ROOT = find_or_create_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


MIN_RUNTIME_PACKAGES = {
    "transformers": "4.56.2",
    "accelerate": "1.10.1",
    "safetensors": "0.4.5",
}


def version_tuple(version: str) -> tuple[int, ...]:
    numbers = [int(part) for part in re.findall(r"\d+", version)[:3]]
    return tuple(numbers + [0] * (3 - len(numbers)))


def ensure_runtime_dependencies() -> None:
    packages_to_install = []
    for package, minimum in MIN_RUNTIME_PACKAGES.items():
        try:
            installed = importlib_metadata.version(package)
        except importlib_metadata.PackageNotFoundError:
            installed = None
        if installed is None or version_tuple(installed) < version_tuple(minimum):
            packages_to_install.append(f"{package}>={minimum}")
            print(f"{package}: {installed or 'not installed'} -> needs >= {minimum}")
        else:
            print(f"{package}: {installed} OK")
    if packages_to_install:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages_to_install])
        raise RuntimeError(
            "Runtime dependencies were installed/upgraded. Restart the kernel, then rerun from Cell 1 "
            "so Python loads the new Transformers registry."
        )


ensure_runtime_dependencies()

from cogmem.consolidation.experiment import load_new_arch_runtime

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
QUANTIZATION_BITS = 0  # Plain GPU weights. This avoids bitsandbytes noise for this behavioral test.
ALLOW_PLAIN_FALLBACK = True
REQUIRED_FREE_VRAM_GB = 8.0
TEMPERATURE = 0.0
MAX_TOKENS = 512

SYSTEM_PROMPT = (
    "You are a precise assistant. Follow the latest user instruction exactly. "
    "Ignore abandoned or explicitly canceled earlier requests."
)

OUTPUT_DIR = Path("/notebooks/cogmem_path_leakage") if Path("/notebooks").exists() else Path("results/path_leakage")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Model:", MODEL_NAME)
print("Quantization bits:", QUANTIZATION_BITS)
print("Temperature:", TEMPERATURE)
print("Output dir:", OUTPUT_DIR)


In [ ]:
# Cell 2: GPU preflight and load local Qwen runtime
import torch


def nvidia_smi_lines(args: list[str]) -> list[str]:
    try:
        output = subprocess.check_output(["nvidia-smi", *args], text=True, stderr=subprocess.STDOUT)
    except Exception as exc:
        print(f"nvidia-smi query failed: {type(exc).__name__}: {exc}")
        return []
    return [line.strip() for line in output.splitlines() if line.strip()]


def gpu_process_rows() -> list[tuple[str, str, str]]:
    rows = []
    for line in nvidia_smi_lines([
        "--query-compute-apps=pid,process_name,used_memory",
        "--format=csv,noheader,nounits",
    ]):
        parts = [part.strip() for part in line.split(",")]
        if len(parts) >= 3:
            rows.append((parts[0], parts[1], parts[2]))
    return rows


def free_vram_gb_from_nvidia_smi() -> float | None:
    lines = nvidia_smi_lines(["--query-gpu=memory.free", "--format=csv,noheader,nounits"])
    values = []
    for line in lines:
        try:
            values.append(float(line) / 1024)
        except ValueError:
            pass
    return max(values) if values else None


def show_gpu_state() -> None:
    rows = gpu_process_rows()
    if not rows:
        print("No active GPU compute processes reported by nvidia-smi.")
        return
    print("GPU processes before load:")
    print("pid, process_name, used_gpu_memory [MiB]")
    for pid, process_name, used_memory in rows:
        print(f"{pid}, {process_name}, {used_memory} MiB")
    pids = " ".join(pid for pid, _, _ in rows)
    print(f"If these are abandoned notebook processes, free VRAM with: !kill -9 {pids}")


for name in ("base_model", "tokenizer", "llm_client"):
    if name in globals():
        del globals()[name]
gc.collect()

show_gpu_state()
free_gb = free_vram_gb_from_nvidia_smi()
if free_gb is not None:
    print(f"Free VRAM before load: {free_gb:.2f} GB")
    if free_gb < REQUIRED_FREE_VRAM_GB:
        raise RuntimeError(
            f"Only {free_gb:.2f} GB VRAM is free. Stop other kernels or restart the Paperspace machine, "
            f"then rerun this cell. Required: {REQUIRED_FREE_VRAM_GB:.1f} GB."
        )

try:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except RuntimeError as exc:
    raise RuntimeError("CUDA is already in an OOM state. Restart this kernel after clearing stale GPU processes.") from exc

runtime_kwargs = {"model_name": MODEL_NAME}
runtime_params = inspect.signature(load_new_arch_runtime).parameters
if "quantization_bits" in runtime_params:
    runtime_kwargs["quantization_bits"] = QUANTIZATION_BITS
if "allow_plain_fallback" in runtime_params:
    runtime_kwargs["allow_plain_fallback"] = ALLOW_PLAIN_FALLBACK

base_model, tokenizer, llm_client = load_new_arch_runtime(**runtime_kwargs)

free_after_gb = free_vram_gb_from_nvidia_smi()
if free_after_gb is not None:
    print(f"Runtime loaded. Free VRAM: {free_after_gb:.1f} GB")
elif torch.cuda.is_available():
    print("Runtime loaded on CUDA")
else:
    print("Runtime loaded on CPU")


In [ ]:
# Cell 3: Experiment definitions
EXPERIMENTS: list[dict[str, Any]] = [
    {
        "id": "exp1_binary_search_path",
        "title": "Experiment 1: same final task, different path",
        "chat_a": ["Write only Python code for iterative binary search. No explanation."],
        "chat_b": [
            "Write Java code for binary search.",
            "No. Ignore that previous request completely.",
            "Write only Python code for iterative binary search. No explanation.",
        ],
        "failure_signs": [
            "Java-like style remains",
            "Adds explanation despite 'No explanation'",
            "Uses recursive implementation",
            "Behavior differs noticeably from Chat A because of earlier Java turn",
        ],
    },
    {
        "id": "exp2_abandoned_intent_leakage",
        "title": "Experiment 2: abandoned intent leakage",
        "chat_a": ["Write a polite email to Professor Kim asking for a meeting tomorrow. Keep it short."],
        "chat_b": [
            "Write an email to Professor Kim asking for research funding.",
            "Forget that. That request is canceled.",
            "Write a polite email to Professor Kim asking for a meeting tomorrow. Keep it short.",
        ],
        "failure_signs": [
            "Mentions funding",
            "Mentions budget",
            "Mentions financial support",
            "Mentions research grant",
        ],
    },
    {
        "id": "exp3_distractor_path",
        "title": "Experiment 3: distractor path",
        "chat_a": ["Summarize Newton's first law in one sentence for a 12-year-old."],
        "chat_b": [
            "Who won the 2018 FIFA World Cup?",
            "Okay. Now summarize Newton's first law in one sentence for a 12-year-old.",
        ],
        "failure_signs": [
            "Answer gets less clear",
            "Answer gets strangely longer",
            "Answer goes off-topic",
            "Football turn lowers final answer quality",
        ],
    },
    {
        "id": "exp4_overwritten_constraints",
        "title": "Experiment 4: overwritten constraints",
        "chat_a": [
            "Give me 3 laptop recommendations for a computer science student under $900. Only list name and price."
        ],
        "chat_b": [
            "Give me 3 gaming laptops over $2000.",
            "Ignore that completely.",
            "Give me 3 laptop recommendations for a computer science student under $900. Only list name and price.",
        ],
        "failure_signs": [
            "Gaming laptop bias remains",
            "Expensive models remain",
            "GPU-heavy recommendations ignore the budget",
            "Prices exceed $900",
        ],
    },
]

print(f"Loaded {len(EXPERIMENTS)} experiments")


In [ ]:
# Cell 4: Runner and check helpers
def run_fresh_chat(user_turns: list[str]) -> dict[str, Any]:
    """Run one fresh chat, preserving generated assistant replies between turns."""
    history: list[dict[str, str]] = [{"role": "system", "content": SYSTEM_PROMPT}]
    turn_outputs: list[dict[str, str]] = []
    final_answer = ""
    for user_text in user_turns:
        history.append({"role": "user", "content": user_text})
        final_answer = llm_client.chat(history, max_tokens=MAX_TOKENS, temperature=TEMPERATURE)
        history.append({"role": "assistant", "content": final_answer})
        turn_outputs.append({"user": user_text, "assistant": final_answer})
    return {"final_answer": final_answer, "turn_outputs": turn_outputs, "message_count": len(history)}


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().lower())


def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()


def contains_any(text: str, terms: list[str]) -> bool:
    lower = text.lower()
    return any(term.lower() in lower for term in terms)


def line_count(text: str) -> int:
    return len([line for line in text.splitlines() if line.strip()])


def price_values(text: str) -> list[int]:
    return [int(match.replace(",", "")) for match in re.findall(r"\$\s*([0-9][0-9,]*)", text)]


def evaluate_flags(exp_id: str, chat_a_answer: str, chat_b_answer: str) -> dict[str, Any]:
    flags: dict[str, Any] = {
        "similarity_to_chat_a": round(similarity(chat_a_answer, chat_b_answer), 3),
        "chat_a_lines": line_count(chat_a_answer),
        "chat_b_lines": line_count(chat_b_answer),
    }
    if exp_id == "exp1_binary_search_path":
        java_terms = ["public static", "class ", "int[]", "boolean ", "System.out", "null"]
        explanation_terms = ["here", "explanation", "this function", "the code", "binary search works"]
        flags.update({
            "b_mentions_java_style": contains_any(chat_b_answer, java_terms),
            "b_has_explanation_markers": contains_any(chat_b_answer, explanation_terms),
            "b_mentions_recursive": contains_any(chat_b_answer, ["recursive", "recursion"]),
            "b_has_python_def": "def " in chat_b_answer,
            "b_has_while_loop": "while " in chat_b_answer,
        })
    elif exp_id == "exp2_abandoned_intent_leakage":
        leak_terms = ["funding", "budget", "financial support", "research grant", "grant"]
        flags.update({
            "b_mentions_canceled_funding_intent": contains_any(chat_b_answer, leak_terms),
            "b_mentions_meeting": contains_any(chat_b_answer, ["meeting", "meet"]),
            "b_mentions_tomorrow": "tomorrow" in chat_b_answer.lower(),
        })
    elif exp_id == "exp3_distractor_path":
        flags.update({
            "b_mentions_football_or_france": contains_any(chat_b_answer, ["france", "fifa", "world cup", "football"]),
            "b_mentions_inertia_or_motion": contains_any(chat_b_answer, ["motion", "moving", "still", "rest", "force"]),
            "b_word_count": len(chat_b_answer.split()),
        })
    elif exp_id == "exp4_overwritten_constraints":
        prices = price_values(chat_b_answer)
        flags.update({
            "b_mentions_gaming_bias": contains_any(chat_b_answer, ["gaming", "rtx", "geforce", "gpu"]),
            "b_prices": prices,
            "b_has_price_over_900": any(price > 900 for price in prices),
            "b_item_count_estimate": max(0, len(re.findall(r"(^|\n)\s*(?:[-*]|\d+[.)])", chat_b_answer))),
        })
    return flags


In [ ]:
# Cell 5: Run all experiments
def run_all_experiments() -> list[dict[str, Any]]:
    results: list[dict[str, Any]] = []
    for idx, exp in enumerate(EXPERIMENTS, start=1):
        print("=" * 80)
        print(f"[{idx}/{len(EXPERIMENTS)}] {exp['title']}")
        print("Running Chat A fresh...")
        chat_a = run_fresh_chat(exp["chat_a"])
        print("Running Chat B fresh...")
        chat_b = run_fresh_chat(exp["chat_b"])
        flags = evaluate_flags(exp["id"], chat_a["final_answer"], chat_b["final_answer"])
        result = {
            "id": exp["id"],
            "title": exp["title"],
            "system_prompt": SYSTEM_PROMPT,
            "temperature": TEMPERATURE,
            "max_tokens": MAX_TOKENS,
            "chat_a_turns": exp["chat_a"],
            "chat_b_turns": exp["chat_b"],
            "chat_a": chat_a,
            "chat_b": chat_b,
            "failure_signs": exp["failure_signs"],
            "flags": flags,
        }
        results.append(result)
        print("Similarity Chat A vs Chat B final:", flags["similarity_to_chat_a"])
        print("Flags:", json.dumps(flags, indent=2, ensure_ascii=False))
        print("\nChat A final:\n", chat_a["final_answer"])
        print("\nChat B final:\n", chat_b["final_answer"])
    return results


results = run_all_experiments()
timestamp = time.strftime("%Y%m%d_%H%M%S")
json_path = OUTPUT_DIR / f"path_leakage_results_{timestamp}.json"
json_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nSaved:", json_path)


In [ ]:
# Cell 6: Compact human-readable summary
def summarize_results(results: list[dict[str, Any]]) -> list[dict[str, Any]]:
    rows = []
    for result in results:
        flags = result["flags"]
        row = {"id": result["id"], "similarity": flags.get("similarity_to_chat_a")}
        if result["id"] == "exp1_binary_search_path":
            row["hard_fail"] = any([
                flags.get("b_mentions_java_style"),
                flags.get("b_has_explanation_markers"),
                flags.get("b_mentions_recursive"),
                not flags.get("b_has_python_def"),
                not flags.get("b_has_while_loop"),
            ])
        elif result["id"] == "exp2_abandoned_intent_leakage":
            row["hard_fail"] = bool(flags.get("b_mentions_canceled_funding_intent"))
        elif result["id"] == "exp3_distractor_path":
            row["hard_fail"] = bool(flags.get("b_mentions_football_or_france"))
        elif result["id"] == "exp4_overwritten_constraints":
            row["hard_fail"] = bool(flags.get("b_mentions_gaming_bias") or flags.get("b_has_price_over_900"))
        rows.append(row)
    return rows


summary_rows = summarize_results(results)
print(json.dumps(summary_rows, indent=2, ensure_ascii=False))

summary_path = OUTPUT_DIR / f"path_leakage_summary_{timestamp}.json"
summary_path.write_text(json.dumps(summary_rows, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved:", summary_path)
